# Rotterdam LST, NDVI, and LCZ calculation

Prerequsites:

```
pip install geemap ee geopandas rasterio rasterstats
```

## Initialise the Project

In [2]:
import os
import ee
import geemap
import geopandas as gpd
import rasterio
from rasterstats import zonal_stats

# ==========================================
# STEP 1: INITIALIZE EARTH ENGINE
# ==========================================

try:
    ee.Initialize(project='applied-spatial-rotterdam')
except Exception:
    ee.Authenticate()
    ee.Initialize(project='applied-spatial-rotterdam')

# ==========================================
# STEP 2: DEFINE AOI & PROCESSING FUNCTIONS
# ==========================================

guangzhou_aoi = ee.Geometry.Rectangle([113.13, 22.85, 113.7, 23.5])

# Summer date ranges to merge into a single multi-year composite
SUMMERS = [
    ('2023-05-01', '2023-08-31'),
    ('2024-05-01', '2024-08-31'),
    ('2025-05-01', '2025-08-31'),
]

def mask_landsat_clouds(image):
    """Mask clouds and cloud shadows using QA_PIXEL band (bits 3 and 4)."""
    qa = image.select('QA_PIXEL')
    mask = qa.bitwiseAnd(1 << 4).eq(0).And(qa.bitwiseAnd(1 << 3).eq(0))
    return image.updateMask(mask)

def process_landsat_indicators(image):
    """Scale optical/thermal bands and derive LST (Celsius) and NDVI."""
    optical = image.select('SR_B.*').multiply(0.0000275).add(-0.2)
    ndvi = optical.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')
    lst_celsius = (
        image.select('ST_B10')
        .multiply(0.00341802).add(149.0).subtract(273.15)
        .rename('LST_Celsius')
    )
    return (
        image
        .addBands(optical, None, True)
        .addBands(lst_celsius, None, True)
        .addBands(ndvi)
    )

def normalize_lst(image):
    """Subtract per-image spatial mean to produce LST anomaly (normalized LST)."""
    lst = image.select('LST_Celsius')
    image_mean = lst.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=guangzhou_aoi,
        scale=30,
        maxPixels=1e9
    ).getNumber('LST_Celsius')
    return (
        lst.subtract(image_mean)
        .rename('LST_Normalized')
        .toFloat()
        .copyProperties(image, image.propertyNames())
    )

## Build Landsat Composite

In [3]:
# ==========================================
# STEP 3: BUILD LANDSAT COMPOSITE
# ==========================================

# Merge all summer scenes across 2023–2025 into a single collection
collections = [
    ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
    .filterBounds(guangzhou_aoi)
    .filterDate(date_start, date_end)
    .filter(ee.Filter.lt('CLOUD_COVER', 30))
    .map(mask_landsat_clouds)
    .map(process_landsat_indicators)
    for date_start, date_end in SUMMERS
]

landsat_collection = collections[0].merge(collections[1]).merge(collections[2])
print(f"Total scenes across 2023–2025 summers: {landsat_collection.size().getInfo()}")

# Single normalized LST composite across all years
lst_normalized = (
    landsat_collection
    .map(normalize_lst)
    .select('LST_Normalized')
    .median()
    .clip(guangzhou_aoi)
)

Total scenes across 2023–2025 summers: 5


## Extract LCZ

In [4]:
# ==========================================
# STEP 4: EXTRACT LOCAL CLIMATE ZONES (LCZ)
# ==========================================

print("Extracting Local Climate Zones...")

lcz_layer = (
    ee.ImageCollection('RUB/RUBCLIM/LCZ/global_lcz_map/latest')
    .mosaic()
    .clip(guangzhou_aoi)
    .select('LCZ_Filter')
)

Extracting Local Climate Zones...


## Write Rasters

In [6]:
# ==========================================
# STEP 5: EXPORT RASTERS
# ==========================================

os.makedirs('../data/guangzhou', exist_ok=True)

ndvi_layer = landsat_collection.median().clip(guangzhou_aoi).select('NDVI')
lst_raster  = '../data/guangzhou/guangzhou_centre_lst_normalized_2023_2025.tif'
ndvi_raster_path = '../data/guangzhou/guangzhou_centre_ndvi_2023_2025.tif'

print("Exporting LCZ raster (100m)...")
geemap.ee_export_image(
    lcz_layer,
    filename='../data/guangzhou/guangzhou_centre_lcz_2018.tif',
    scale=100,
    region=guangzhou_aoi,
    file_per_band=False
)

print("Exporting normalized LST raster (30m)...")
geemap.ee_export_image(
    lst_normalized,
    filename=lst_raster,
    scale=30,
    region=guangzhou_aoi,
    file_per_band=False
)

print("Exporting NDVI raster (30m)...")
geemap.ee_export_image(
    ndvi_layer,
    filename=ndvi_raster_path,
    scale=30,
    region=guangzhou_aoi,
    file_per_band=False
)

print("All rasters exported to ../data/processed/")

Exporting LCZ raster (100m)...
Generating URL ...
Please wait ...
Data downloaded to C:\Users\artem\Documents\TUDelft\ARFW0501\report\data\guangzhou\guangzhou_centre_lcz_2018.tif
Exporting normalized LST raster (30m)...
Generating URL ...
Please wait ...
Data downloaded to C:\Users\artem\Documents\TUDelft\ARFW0501\report\data\guangzhou\guangzhou_centre_lst_normalized_2023_2025.tif
Exporting NDVI raster (30m)...
Generating URL ...
Please wait ...
Data downloaded to C:\Users\artem\Documents\TUDelft\ARFW0501\report\data\guangzhou\guangzhou_centre_ndvi_2023_2025.tif
All rasters exported to ../data/processed/


# Guangzhou Roads

## Get Guangzhou Roads

In [7]:
import geopandas as gpd
import osmnx as ox
from shapely.ops import unary_union

# Geocode each district individually
districts = [
    "Huangpu District, Guangzhou, China",
    "Tianhe District, Guangzhou, China",
    "Yuexiu District, Guangzhou, China",
    "Baiyun District, Guangzhou, China",
    "Liwan District, Guangzhou, China",
    "Panyu District, Guangzhou, China",
    "Haizhu District, Guangzhou, China",
]

district_gdfs = [ox.geocode_to_gdf(d) for d in districts]

# Union all district polygons in a projected CRS, then reproject back
district_union = unary_union(
    [gdf.to_crs("EPSG:4497").geometry.iloc[0] for gdf in district_gdfs]
)

# Buffer the union by 1 000 m (still in EPSG:4491), then back to WGS 84
union_gdf = gpd.GeoDataFrame(geometry=[district_union], crs="EPSG:4497")
union_buffered_wgs84 = union_gdf.buffer(1000).to_crs("EPSG:4326").iloc[0]

G = ox.graph_from_polygon(union_buffered_wgs84, network_type="walk")

nodes, edges = ox.graph_to_gdfs(G)
nodes = nodes.to_crs("EPSG:4497")
edges = edges.to_crs("EPSG:4497")

nodes.to_file(
    "../data/guangzhou/guangzhou_roads.gpkg", layer="nodes", driver="GPKG"
)
edges.to_file(
    "../data/guangzhou/guangzhou_roads.gpkg", layer="edges", driver="GPKG"
)

print(ox.basic_stats(G))

{'n': 88460, 'm': 254992, 'k_avg': 5.765136784987565, 'edge_length_total': 32922468.063772738, 'edge_length_avg': 129.11176846243308, 'streets_per_node_avg': 2.8887632828397014, 'streets_per_node_counts': {0: 0, 1: 12900, 2: 0, 3: 60140, 4: 14949, 5: 411, 6: 55, 7: 2, 8: 2, 9: 1}, 'streets_per_node_proportions': {0: 0.0, 1: 0.1458286231064888, 2: 0.0, 3: 0.6798553018313362, 4: 0.16899163463712413, 5: 0.004646167759439295, 6: 0.0006217499434772779, 7: 2.2609088853719196e-05, 8: 2.2609088853719196e-05, 9: 1.1304544426859598e-05}, 'intersection_count': 75560, 'street_length_total': 16461209.844186146, 'street_segment_count': 127495, 'street_length_avg': 129.1125914285748, 'circuity_avg': 1.0851164581428312, 'self_loop_proportion': 0.001568688968194831}


**_IMPORTANT_**

**Before** executing next code, you should modify `guangzhou_roads` in QGIS/PST.

We recommend that you:
- Start a project with a _metric_ CRS (e.g. EPSG:28992)
- Upload the `edges` layer from `guangzhou_roads.gpkg`
- Open the processing toolbox and perform the following operations:
    - Split with lines (Input layer: `guangzhou_roads — edges`, Split layer: `guangzhou_roads — edges`)
    - Multipart to singleparts
    - If necessary, Fix geometries
- Open the PST plugin and run:
    - Create segment map
    - Perform Network Betweenness
        - Distance modes: walking distance
        - Weight modes: no weight
        - Normalisation modes: no normalisation
        - Radius: walking distance, 1000m
- Save the resulting file as `guangzhou_roads_PST.gpkg`, with the layer name `guangzhou_roads_PST`.